# ¿Cuánto durará la fiebre de GTA VI?

**Roberto Cantero · Nebula · Machine learning experimental**

## Mi objetivo y mi protocolo

Estimo la media diaria de consultas en las semanas tercera y quinta, como fracción de las consultas del día de estreno. No estimo cuándo se deja de jugar. Mi protocolo está en protocolo.md, escrito antes de calcular el modelo. Uso dos señales disponibles al cuarto día: la caída entre el estreno y el día +3, y el interés previo relativo al estreno. Ajusto una regresión ridge sobre el logaritmo de la retención. Regularizo para limitar ajustes excesivos con pocos juegos.

In [1]:
# reproduzco mi preparación y mi ajuste
from modelo import ejecutar, preparar, ajustar, predecir, referencia, medir
import numpy as np
import json
r=ejecutar()
juegos,gta=preparar()
print(json.dumps(r['muestra'],ensure_ascii=False,indent=2))
print('Mi penalización elegida solo en validación:',r['alpha_elegido'])

{
  "recogidos": 30,
  "utilizados": 28,
  "entrenamiento": 14,
  "validacion": 7,
  "prueba": 7,
  "exclusiones": {
    "Dark_Souls_III": "estreno japonés anterior al mundial",
    "Baldur's_Gate_3": "cobertura incompleta del título en los primeros días"
  }
}
Mi penalización elegida solo en validación: 10.0


## Cómo compruebo si mi modelo aporta algo

Entreno con catorce juegos hasta 2019. Elijo entre dos penalizaciones con siete juegos de 2020–2021. Después reajusto con esos veintiuno y evalúo una vez en siete juegos de 2022–2023. Mantengo cada videojuego entero en su grupo. No mezclo días del mismo juego entre entrenamiento y prueba. Mi criterio de selección es el error absoluto medio del logaritmo de retención: penaliza desviaciones proporcionales. Lo acompaño con errores en puntos porcentuales, más fáciles de interpretar.

Mis referencias son conservar el nivel del cuarto día y aplicar la mediana histórica de retención de cada horizonte. Esta segunda referencia ya sabe que la atención suele caer: superarla es una comparación más exigente.

In [2]:
# examino mi validación sin usar la prueba para escoger la penalización
print(json.dumps(r['seleccion_validacion'],ensure_ascii=False,indent=2))
print('Mi prueba final:')
print(json.dumps(r['metricas_prueba'],ensure_ascii=False,indent=2))
print('Mi mejora relativa en el error logarítmico:',round(100*r['mejora_relativa_mae_log'],2),'%')
print('Resultados por juego:')
for s in r['prueba_por_juego']:
    print(s['titulo'], 'real',np.round(100*np.array(s['real']),2), 'modelo',np.round(100*np.array(s['ridge']),2),'mediana',np.round(100*np.array(s['mediana']),2))

{
  "1.0": {
    "mae_log": 0.4676575960934371,
    "mae_log_horizonte": [
      0.34723935207562745,
      0.5880758401112468
    ],
    "mae_puntos_retencion": [
      5.277646397300771,
      5.046769089312864
    ],
    "factor_error_geometrico": 1.5962507466123117
  },
  "10.0": {
    "mae_log": 0.46433769689799814,
    "mae_log_horizonte": [
      0.3605198235744115,
      0.5681555702215848
    ],
    "mae_puntos_retencion": [
      5.488564937956132,
      4.823689589057759
    ],
    "factor_error_geometrico": 1.5909601420391482
  }
}
Mi prueba final:
{
  "ridge": {
    "mae_log": 0.15768193072681008,
    "mae_log_horizonte": [
      0.1431963086392516,
      0.1721675528143686
    ],
    "mae_puntos_retencion": [
      2.2562215331661233,
      1.482889016032089
    ],
    "factor_error_geometrico": 1.1707937419633612
  },
  "mediana": {
    "mae_log": 0.18827799643988566,
    "mae_log_horizonte": [
      0.18157249409691342,
      0.19498349878285795
    ],
    "mae_puntos_r

## Lo que encuentro en mi prueba

Mi modelo mejora un 16,25 % el error logarítmico de la mediana histórica en siete juegos de prueba. El error absoluto medio es de 2,26 puntos de retención para la tercera semana y 1,48 para la quinta. Son puntos sobre el nivel del estreno, no un error relativo del 2,26 % o 1,48 %. La prueba es pequeña y no demuestra una mejora universal. No atribuyo significación estadística al resultado. El conjunto de validación fue más difícil; no uso solo el buen resultado final para esconderlo.

In [3]:
# aplico el mismo modelo evaluado sin añadir datos futuros de gta vi
print(json.dumps(r['gta'],ensure_ascii=False,indent=2))
assert r['supera_referencia']
assert r['gta']['periodos'][0]['inicio']=='2026-12-03'
assert r['gta']['periodos'][1]['fin']=='2026-12-23'
print('Mi modelo usa solo los cuatro primeros días observados de GTA VI y la semana anterior.')

{
  "dia0": 750227,
  "dia3": 308867,
  "cambio_observado_pct": -58.83019406126413,
  "prediccion_experimental_retencion": [
    0.12284668958784808,
    0.06848940600498692
  ],
  "prediccion_experimental_consultas": [
    92162.9033894225,
    51382.60159890332
  ],
  "rango_orientativo_retencion": [
    [
      0.060717192658315856,
      0.24855083843581147
    ],
    [
      0.021628742291770767,
      0.21687801683691418
    ]
  ],
  "rango_orientativo_consultas": [
    [
      45551.67729647033,
      186469.54986718352
    ],
    [
      16226.466443328307,
      162707.74393750762
    ]
  ],
  "rango_no_probabilistico": true,
  "periodos": [
    {
      "inicio": "2026-12-03",
      "fin": "2026-12-09"
    },
    {
      "inicio": "2026-12-17",
      "fin": "2026-12-23"
    }
  ],
  "variables": [
    -0.8874650638745194,
    -1.5787652590071315
  ],
  "rango_variables_entrenamiento": [
    [
      -0.8489438745983421,
      -1.6475626063632434
    ],
    [
      -0.2293082939

## Mi previsión experimental y su incertidumbre

Estimo alrededor de 92.000 consultas diarias para el 3–9 de diciembre, un 12,3 % del día de estreno. Para el 17–23 de diciembre obtengo unas 51.000, un 6,8 %. Mi banda orientativa va de unas 46.000 a 186.000 consultas en la primera ventana y de 16.000 a 163.000 en la segunda. La construyo aplicando a la previsión el mayor error logarítmico de mis siete juegos de validación para cada horizonte.

No es un intervalo con cobertura del 80 %, 90 % o 95 % garantizada. Reajusto después de esa validación y GTA VI puede no ser intercambiable con los juegos anteriores. Las bandas son una traducción de errores históricos a escala de GTA VI, no probabilidades calibradas.

El estreno de GTA VI supera en consultas al mayor estreno de mi entrenamiento y validación. Además, su caída inicial, 58,8 %, es algo mayor que la más intensa de esos veintiún juegos. Estoy extrapolando ligeramente una variable; normalizar las curvas no elimina ese límite. No interpreto la magnitud del estreno como una cantidad de jugadores.

In [4]:
# compruebo mi extrapolación y comparo con el registro más sencillo
previos=[s for s in juegos if s['grupo']!='prueba']
print('Mayor estreno previo:',max(s['dia0'] for s in previos),'frente a GTA VI:',gta['dia0'])
print('Caída inicial de GTA VI:',round(100*(gta['dia3']/gta['dia0']-1),2),'%')
print('Mayor caída inicial en mis datos de ajuste:',round(100*(min(s['dia3']/s['dia0'] for s in previos)-1),2),'%')
print('Referencia de mediana aplicada a GTA VI:',np.round(referencia(previos,[gta],'mediana')[0]*gta['dia0']))
print('Mi prueba no contiene los datos futuros de GTA VI: todavía no han ocurrido.')

Mayor estreno previo: 436767 frente a GTA VI: 750227
Caída inicial de GTA VI: -58.83 %
Mayor caída inicial en mis datos de ajuste: -57.21 %
Referencia de mediana aplicada a GTA VI: [121339.  65089.]
Mi prueba no contiene los datos futuros de GTA VI: todavía no han ocurrido.


## Mi conclusión y cómo contrastarla después

Mi previsión es experimental: la mejora sobre la mediana existe en esta prueba, pero es modesta y la muestra es pequeña. Publico tanto la cifra central como un rango amplio. No predigo ventas, jugadores, calidad ni el fin del éxito de GTA VI. Mi predicción se refiere a las consultas de una página concreta.

Después del 9 y del 23 de diciembre descargaré las ventanas completas, conservaré esta previsión sin modificarla y calcularé su error. Si cambian títulos o redirecciones, documentaré el cambio en vez de sustituir silenciosamente la serie. Las instrucciones de repetición y las peticiones están en fuentes_metodologia.md. Mi código solo necesita NumPy y los archivos conservados.